# Interacting with Language Models

## Interacting with LLMs via APIs or SDKs

Large Language Models (LLMs) can be accessed and utilized through APIs or SDKs provided by their respective platforms. APIs allow developers to send requests to the model hosted on the provider's servers and receive responses, while SDKs often provide additional tools and utilities to simplify integration into applications.

For example, Google provides access to its Gemini model through its API. By using the Gemini model, developers can leverage advanced natural language understanding and generation capabilities for tasks such as summarization, classification, and more. To interact with the Gemini model, you would typically authenticate with an API key, send a request with the desired input, and process the response returned by the model.

## 1. Installing the SDK and Importing Libraries

In this section, we will install the required SDKs and import the necessary libraries to interact with the OpenAI API.

In [ ]:
# Install required packages - note: use spaces, not commas between package names
!pip install python-dotenv google-genai openai requests

In [ ]:
import requests
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from google import genai
from google.genai import types

from IPython.display import HTML, Markdown, display

## 2. Setup API Key

To get your API key from Google AI Studio (for Gemini models), follow these steps:

- Visit: https://makersuite.google.com/app

> If you don't have access yet, you may need to request access or sign in with a supported Google account.

- Select "Get API Key" or go directly to: https://aistudio.google.com/app/apikey

- Click on "Create API Key" if you don't have one already.

- Copy the generated key in a .env file:
API_KEY=SECRET_KEY_FROM_GOOGLE
> 💡 Keep it secret: Never share your API key publicly or hard-code it in shared scripts.


In [ ]:
# Load environment variables from the .env file
load_dotenv()

# Set the API key and model information
API_KEY = os.getenv("API_KEY")
API_URL = 'https://generativelanguage.googleapis.com/v1beta/openai'
MODEL = "gemini-2.0-flash"

# Check if API key is available
if not API_KEY:
    print("⚠️ Warning: API_KEY not found in .env file. Please add your API key to continue.")

## 2.1 Making API Requests with the `requests` Library

To interact with the Gemini API using the `requests` library, follow these steps:

1. **Set up the headers**: Include the API key for authentication and specify the content type as JSON.
2. **Define the payload**: Construct the request body with the required parameters, such as the model and messages.
3. **Send the request**: Use the `requests.post` method to send a POST request to the API endpoint.
4. **Handle the response**: Check the response status code and parse the JSON response to extract the desired information.

For more details on the Gemini API, refer to the [Gemini API Reference Documentation](https://ai.google.dev/gemini-api/docs/openai).


In [ ]:
# Define the headers and payload
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

data = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a detailed explanation of the insurance underwriting process"}
    ]
}

# Make the POST request
try:
    response = requests.post(
        API_URL + '/chat/completions',
        headers=headers, json=data)
    
    # Check the response and print the result
    response.raise_for_status()  # Raise an exception for HTTP errors
    response_json = response.json()
    print("✅ API request successful!")
    print(response_json['choices'][0]['message']['content'][:100] + "...")
except requests.exceptions.RequestException as e:
    print(f"❌ API request failed: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print(f"Error details: {e.response.text}")

The API response is returned in JSON format and typically contains the following key components:

1. **`choices`**:
   - This is a list of possible completions generated by the model.
   - Each item in the list contains:
     - **`message`**: The actual content of the response.
     - **`finish_reason`**: Indicates why the generation stopped (e.g., "stop", "length").
     - **`index`**: The index of the completion in the list.

2. **`usage`**:
   - Provides metadata about token usage:
     - **`prompt_tokens`**: The number of tokens used in the input prompt.
     - **`completion_tokens`**: The number of tokens generated in the response.
     - **`total_tokens`**: The total number of tokens used (prompt + completion).

3. **`model`**:
   - Specifies the model used to generate the response (e.g., `gpt-4`, `gemini-2.0-flash`).


In [ ]:
# Display the full response
response_json

In [ ]:
# Display the response as formatted Markdown for better readability
if 'response_json' in locals():
    display(Markdown(response_json['choices'][0]['message']['content']))
else:
    display(Markdown("*No valid response available. Please run the previous cell successfully first.*"))

## Using Existing SDKs for Interacting with APIs

In addition to using the `requests` library to make API calls, many platforms provide SDKs (Software Development Kits) that simplify the process of interacting with their APIs. SDKs often include pre-built methods and utilities, reducing the need for manual setup and handling of requests and responses.

For example, in this notebook, we are using the `openai` SDK to interact with the Gemini model. By leveraging the SDK, we can directly call methods like `client.chat.completions.create()` to send requests and receive responses, without needing to manually construct headers or payloads.

### Benefits of Using SDKs:
- **Ease of Use**: SDKs abstract away the complexity of API calls, making it easier to integrate with the service.
- **Error Handling**: SDKs often include built-in error handling and debugging tools.
- **Consistency**: SDKs ensure that your requests are formatted correctly and comply with the API's requirements.
- **Additional Features**: SDKs may provide additional utilities, such as authentication helpers, data parsing, and more.

For more details on using the `openai` SDK, refer to the [OpenAI SDK Documentation](https://platform.openai.com/docs/). Similarly, other platforms like Google AI Studio provide their own SDKs for interacting with models like Gemini.

In [ ]:
# Using the OpenAI SDK to interact with Gemini API
try:
    # Initialize the OpenAI client with our API key and base URL
    client = OpenAI(
        api_key=API_KEY,
        base_url=API_URL 
    )

    # Create a chat completion using the SDK
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": "Write a detailed explanation of the insurance underwriting process"
            }
        ]
    )
    
    # Display the response as formatted Markdown
    display(Markdown(response.choices[0].message.content))
    
except Exception as e:
    print(f"❌ Error using OpenAI SDK: {str(e)}")

In [ ]:
# Using the Google Generative AI SDK to interact with Gemini API
try:
    # Initialize the Google Generative AI client
    google_client = genai.Client(api_key=API_KEY)

    # Generate content using the SDK
    response = google_client.models.generate_content(
        model=MODEL,
        contents='Write a detailed explanation of the insurance underwriting process')

    # Display the response as formatted Markdown
    display(Markdown(response.text))
    
except Exception as e:
    print(f"❌ Error using Google Generative AI SDK: {str(e)}")

## 3. Explore Generation Parameters

### 3.1 Output Length

When generating text with an LLM, the output length affects cost and performance. Generating more tokens increases computation, leading to higher energy consumption, latency, and cost.

To stop the model from generating tokens past a limit, you can specify the `max_completion_tokens` parameter that limits the number of tokens (words or parts of words) the AI model can generate in its response.



In [ ]:
# Demonstrate restricting output length with max_completion_tokens
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Generate a 1000-word content for an underwriting application."}],
        max_completion_tokens=200  # Limit response to approximately 200 tokens
    )
    print("Response limited to 200 tokens:")
    print(response.choices[0].message.content)
    
    # Print approximate token count (rough estimate based on words)
    word_count = len(response.choices[0].message.content.split())
    print(f"\nApproximate word count: {word_count}")
    
except Exception as e:
    print(f"❌ Error limiting output length: {str(e)}")

## 3.2 Controlling Randomness and Diversity

When generating text, you can control the randomness and diversity of the output using parameters like `temperature`, `top_k`, and `top_p`.

### Temperature
- The `temperature` parameter controls the randomness of the model's output.
- A higher value (e.g., 1.0) results in more random and creative responses, while a lower value (e.g., 0.2) makes the output more focused and deterministic.
- At temperature=0.0, the model will always choose the most probable next token (greedy decoding).

### Top-p Sampling (Nucleus Sampling)
- The `top_p` parameter ensures that the model considers only the smallest set of tokens whose cumulative probability exceeds `p`.
- For example, if `top_p=0.9`, the model will sample from the smallest set of tokens that together have a 90% probability, dynamically adjusting the number of tokens considered.

### Top-k Sampling
- The `top_k` parameter limits the model to consider only the top `k` most probable tokens at each step.
- For example, if `top_k=50`, the model will only sample from the 50 most likely tokens, reducing the chance of selecting less probable tokens.

> Top-K is not configurable in the Gemini 2.0 series of models, but can be changed in older models.

These parameters can be used individually or in combination to fine-tune the balance between randomness and coherence in the generated text.

In [ ]:
# Demonstrate the effect of low temperature (deterministic output)
try:
    # Configure temperature = 0.0 for deterministic outputs
    low_temp_config = types.GenerateContentConfig(temperature=0.0)
    google_client = genai.Client(api_key=API_KEY)

    print("🌡️ Temperature = 0.0 (deterministic outputs):")
    print("Asking for a random color 5 times - results should be consistent")
    print("-" * 50)
    
    for i in range(5):
        response = google_client.models.generate_content(
            model=MODEL,
            config=low_temp_config,
            contents='Pick a random colour... (respond in a single word)')

        if response.text:
            print(f"Run {i+1}: {response.text}")

    # Now demonstrate high temperature (more randomness)
    print("\n🌡️ Temperature = 1.0 (more random outputs):")
    print("Asking for a random color 5 times - results should vary")
    print("-" * 50)
    
    high_temp_config = types.GenerateContentConfig(temperature=1.0)
    
    for i in range(5):
        response = google_client.models.generate_content(
            model=MODEL,
            config=high_temp_config,
            contents='Pick a random colour... (respond in a single word)')

        if response.text:
            print(f"Run {i+1}: {response.text}")
            
except Exception as e:
    print(f"❌ Error demonstrating temperature: {str(e)}")

In [ ]:
# Demonstrate combined temperature and top_p settings
try:
    # Configure model with specific generation parameters
    model_config = types.GenerateContentConfig(
        temperature=1.0,  # High creativity
        top_p=0.95,       # Consider tokens with 95% cumulative probability
    )

    story_prompt = "You are a creative writer. Write a short story about a cat who goes on an adventure."
    response = google_client.models.generate_content(
        model=MODEL,
        config=model_config,
        contents=story_prompt)

    print("🐱 Creative Story with temperature=1.0, top_p=0.95:")
    print("-" * 50)
    print(response.text)
    
    # Let's generate another story with different parameters for comparison
    conservative_config = types.GenerateContentConfig(
        temperature=0.3,  # Lower creativity, more focused
        top_p=0.5,        # Only consider higher probability tokens
    )
    
    response2 = google_client.models.generate_content(
        model=MODEL,
        config=conservative_config,
        contents=story_prompt)
    
    print("\n🐱 More Focused Story with temperature=0.3, top_p=0.5:")
    print("-" * 50)
    print(response2.text)
    
except Exception as e:
    print(f"❌ Error with generation parameters: {str(e)}")

## 4. Prompting

This section will introduce you to the concept of prompting.

Prompting is the process of providing a model with a specific input or instruction to generate a desired output. The quality and specificity of your prompt can significantly influence the model's response.

There are several prompting techniques, each with its own advantages:

### 4.1 Zero-Shot Prompting

Zero-shot prompting involves asking the model to perform a task without providing any examples. The model relies on its pre-training to understand and complete the task.

In [ ]:
# Zero-shot prompt with JSON format
zero_shot_prompt = """
Extract the information from the following application for risk assessment and return it as a JSON object:
Age: 22, Smoker: Yes, BMI: 32, Occupation: Construction Worker.

Output the result in JSON format.
"""

# Call API here (pseudo-code)
# response = call_model(prompt)
# print(response)

### 4.2 Few-shot Prompting

Few-shot prompting involves providing the model with a few examples of the task before asking it to perform the same task with new input. This approach can improve the model's understanding of the task and lead to more accurate outputs.

In [ ]:
# Few-shot prompt with JSON format
few_shot_prompt = """
Extract the information from the following applications for risk assessment and return it as a JSON object:

Example 1:
Input: Age: 30, Smoker: No, BMI: 25, Occupation: Teacher
Output:
{
  "age": 30,
  "smoker": "No",
  "bmi": 25,
  "occupation": "Teacher"
}

Example 2:
Input: Age: 45, Smoker: Yes, BMI: 28, Occupation: Engineer
Output:
{
  "age": 45,
  "smoker": "Yes",
  "bmi": 28,
  "occupation": "Engineer"
}

Now, extract the information from this application:
Input: Age: 22, Smoker: Yes, BMI: 32, Occupation: Construction Worker
Output:
"""

# response = call_chat_model(messages)
# print(response)

### 4.3 Chain-of-Thought (CoT) Prompting

Direct prompting with large language models (LLMs) can generate fast and token-efficient responses. However, this approach is more susceptible to hallucinations—answers that appear linguistically and syntactically correct but are factually or logically inaccurate.

**Chain-of-Thought (CoT)** prompting mitigates this by guiding the model to produce intermediate reasoning steps. This often leads to more accurate results, particularly when used with few-shot examples. However, CoT does not fully eliminate hallucinations and comes with a higher token cost due to the additional reasoning content.

In [ ]:
# CoT prompt - encouraging step-by-step reasoning
cot_prompt = """
Extract the information from the following applications for risk assessment and return it as a JSON object. Think step by step to ensure accuracy.

Example 1:
Input: Age: 30, Smoker: No, BMI: 25, Occupation: Teacher
Step-by-step reasoning:
1. Identify the age: 30
2. Identify if the person is a smoker: No
3. Identify the BMI: 25
4. Identify the occupation: Teacher
5. Format the information into a JSON object.
Output:
{
  "age": 30,
  "smoker": "No",
  "bmi": 25,
  "occupation": "Teacher"
}

Now, extract the information from this application:
Input: Age: 22, Smoker: Yes, BMI: 32, Occupation: Construction Worker
Step-by-step reasoning:
"""

# response = call_model(prompt)
# print(response)

To learn more about prompting, please visit https://learnprompting.org/docs/introduction for detailed explanations and examples.

## 5. Structure LLM Output

This section demonstrates how to structure LLM outputs using different techniques, including function calling, JSON output, and structured output.

### 5.1 Function Calling

Function calling allows LLMs to generate structured outputs by invoking predefined functions. This is useful for tasks like data extraction or executing specific operations.

In [ ]:
# Define a function schema for extracting user profiles
openai_function = {
    "type": "function",
    "function": {
        "name": "extract_user_profile",
        "description": "Extracts user profile information.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "User's name."},
                "age": {"type": "integer", "description": "User's age."},
                "email": {"type": "string", "description": "User's email address."}
            },
            "required": ["name", "age", "email"]
        }
    }
}

try:
    # Note: Function calling requires an OpenAI model or compatible model
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Extract user profile information."},
            {"role": "user", "content": "Name: John Doe, Age: 30, Email: john.doe@example.com"}
        ],
        tools=[openai_function]
    )
    
    # Check if a function was called
    if completion.choices[0].message.tool_calls:
        function_args = completion.choices[0].message.tool_calls[0].function.arguments
        print("Function calling result:")
        print(function_args)
        
        # Parse and prettify the JSON for better visualization
        try:
            parsed_json = json.loads(function_args)
            print("\nFormatted JSON:")
            print(json.dumps(parsed_json, indent=2))
        except json.JSONDecodeError:
            print("\nWarning: Function arguments could not be parsed as JSON")
    else:
        print("No function was called in the response.")
        
except Exception as e:
    print(f"❌ Error with function calling: {str(e)}")
    print("Note: Function calling might not be supported by all models or configurations.")

### 5.2 JSON Output

LLMs can directly generate JSON-formatted outputs, which are easy to parse and integrate into applications. You can request JSON output using the `response_format` parameter.

In [ ]:
# Requesting explicit JSON output
json_prompt = """
Extract the following user profile information and return it as a JSON object:
Name: Jane Doe, Age: 28, Email: jane.doe@example.com
"""

# Set response_format to request JSON output
completion = client.chat.completions.create(
    model=MODEL,  # Note: Using a different model as Gemini might not fully support this feature
    messages=[
        {"role": "system", "content": "Extract user profile information and return it as JSON."},
        {"role": "user", "content": json_prompt}
    ],
    response_format={"type": "json_object"},
)

# Print the raw JSON response
print("JSON output result:")
print(completion.choices[0].message.content)


### 5.3 Structured Output with Pydantic

Using Pydantic models, you can validate and parse structured outputs from LLMs, ensuring the data adheres to a predefined schema. This is especially useful for type validation and integration with larger systems.

In [ ]:
# Install pydantic if not already installed
try:
    from pydantic import BaseModel
except ImportError:
    !pip install pydantic
    from pydantic import BaseModel

In [ ]:
# Define a Pydantic model for user profiles
class UserProfile(BaseModel):
    name: str
    age: int
    email: str

# Create a prompt that describes the expected structure
structured_prompt = """
Extract the following user profile information:
Name: Jane Doe, Age: 28, Email: jane.doe@example.com
Return the result as a JSON object with fields for name, age, and email.
"""

# Get the JSON schema from the Pydantic model
schema_info = UserProfile.model_json_schema()
schema_str = json.dumps(schema_info, indent=2)

# Add the schema to the system message
system_message = f"Extract user profile information using this schema: {schema_str}"

# Request a JSON response that matches our schema
parsed = client.beta.chat.completions.parse(
    model=MODEL,  # Note: Using a different model as Gemini might not fully support this feature
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": structured_prompt},
    ],
    response_format=UserProfile
)


In [ ]:
parsed

In [ ]:
# Get the result as a Pydantic model
user_profile = UserProfile.model_validate_json(parsed.choices[0].message.content)
user_profile